# FedFalsify v0.6 — Resumable Colab Confirmatory Experiment

This notebook runs the frozen confirmatory matrix from GitHub, mirrors every
completed chunk to Google Drive, and can optionally commit/push the result files
back to GitHub.

**Scientific safeguards**

- Do not replace failed seeds.
- Do not tune the algorithm after seeing confirmatory results.
- Run all four chunks with the same configuration.
- Use `merge` only after all chunks are complete.
- A GPU is not required; use a CPU or high-RAM runtime.

In [ ]:
#@title 1. Configuration
REPO_URL = "https://github.com/AzizulHakim00/fedfalsify.git" #@param {type:"string"}
CODE_BRANCH = "feat/fedfalsify-mvi" #@param {type:"string"}
PUSH_BRANCH = "feat/fedfalsify-mvi" #@param {type:"string"}
RUN_ID = "v06-primary-confirmatory" #@param {type:"string"}

MODE = "dry_run" #@param ["dry_run", "primary_chunk", "merge"]
CHUNK_INDEX = 0 #@param {type:"integer"}
TOTAL_CHUNKS = 4 #@param {type:"integer"}

DRIVE_ROOT = "/content/drive/MyDrive/FedFalsify/results" #@param {type:"string"}
PUSH_TO_GITHUB = True #@param {type:"boolean"}
PUSH_DRY_RUN = False #@param {type:"boolean"}

GIT_USER_NAME = "Azizul Hakim" #@param {type:"string"}
GIT_USER_EMAIL = "" #@param {type:"string"}

assert MODE in {"dry_run", "primary_chunk", "merge"}
assert TOTAL_CHUNKS >= 1
assert 0 <= CHUNK_INDEX < TOTAL_CHUNKS
print({
    "mode": MODE,
    "run_id": RUN_ID,
    "chunk_index": CHUNK_INDEX,
    "total_chunks": TOTAL_CHUNKS,
    "drive_root": DRIVE_ROOT,
})

In [ ]:
#@title 2. Mount Drive, clone GitHub, and install
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_DIR = Path("/content/fedfalsify")
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "checkout", CODE_BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", CODE_BRANCH], check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", CODE_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
    os.chdir(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], check=True)
subprocess.run(["git", "config", "user.name", GIT_USER_NAME], check=True)
if GIT_USER_EMAIL.strip():
    subprocess.run(["git", "config", "user.email", GIT_USER_EMAIL.strip()], check=True)

print("Repository:", REPO_DIR)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
subprocess.run(["fedfalsify-colab", "--help"], check=True)

In [ ]:
#@title 3. Validate the installation
import subprocess
subprocess.run(["pytest", "-q", "tests/test_colab_pipeline.py"], check=True)
print("Colab pipeline validation passed.")

## Run modes

### `dry_run`
Runs one small technical validation. It uses a separate run ID and does not
change the frozen primary results.

### `primary_chunk`
Runs one quarter of seeds `9001–9020`. Run chunk indexes `0`, `1`, `2`, and `3`.
Each completed chunk is saved under:

- Git working tree: `results/colab/v06-primary-confirmatory/chunks/...`
- Google Drive: `MyDrive/FedFalsify/results/v06-primary-confirmatory/chunks/...`

### `merge`
Restores Drive chunks into the Git working tree, checks that all four chunks
exist, verifies exactly 2,400 unique method-condition rows, recomputes the final
summary and Holm correction, and writes the final outputs.

In [ ]:
#@title 4. Run the selected mode
from pathlib import Path
import shutil
import subprocess

repo_output_root = REPO_DIR / "results" / "colab"
drive_output_root = Path(DRIVE_ROOT)
drive_output_root.mkdir(parents=True, exist_ok=True)

if MODE == "dry_run":
    active_run_id = f"{RUN_ID}-dry-run"
    command = [
        "fedfalsify-colab", "run-chunk",
        "--run-id", active_run_id,
        "--output-root", str(repo_output_root),
        "--drive-root", str(drive_output_root),
        "--benchmarks", "base",
        "--scenarios", "complementary",
        "--noise", "0.03",
        "--samples", "60",
        "--clients", "4",
        "--seeds", "9001",
        "--chunk-index", "0",
        "--total-chunks", "1",
        "--population-size", "12",
        "--generations", "2",
        "--max-genes", "3",
        "--bootstrap-resamples", "500",
    ]
elif MODE == "primary_chunk":
    active_run_id = RUN_ID
    command = [
        "fedfalsify-colab", "run-chunk",
        "--run-id", active_run_id,
        "--output-root", str(repo_output_root),
        "--drive-root", str(drive_output_root),
        "--benchmarks", "base,poly3,nested_sine,trig_product,interaction",
        "--scenarios", "complementary,spurious,exception",
        "--noise", "0.03,0.10",
        "--samples", "300",
        "--clients", "4",
        "--seeds", "9001-9020",
        "--chunk-index", str(CHUNK_INDEX),
        "--total-chunks", str(TOTAL_CHUNKS),
        "--population-size", "48",
        "--generations", "12",
        "--max-genes", "4",
        "--bootstrap-resamples", "4000",
    ]
else:
    active_run_id = RUN_ID
    # Restore any chunks that exist only in Drive before validating the merge.
    drive_run = drive_output_root / RUN_ID
    repo_run = repo_output_root / RUN_ID
    if drive_run.exists():
        shutil.copytree(drive_run, repo_run, dirs_exist_ok=True)
    command = [
        "fedfalsify-colab", "merge",
        "--run-id", RUN_ID,
        "--output-root", str(repo_output_root),
        "--drive-root", str(drive_output_root),
        "--expected-chunks", str(TOTAL_CHUNKS),
        "--expected-rows", "2400",
        "--bootstrap-resamples", "10000",
    ]

print("Executing:", " ".join(command))
subprocess.run(command, check=True, cwd=REPO_DIR)
print("Mode completed:", MODE)

In [ ]:
#@title 5. Inspect saved files
from pathlib import Path
import json

active_run_id = f"{RUN_ID}-dry-run" if MODE == "dry_run" else RUN_ID
repo_run = REPO_DIR / "results" / "colab" / active_run_id
drive_run = Path(DRIVE_ROOT) / active_run_id

print("Git working-tree output:", repo_run)
print("Drive mirror:", drive_run)

for path in sorted(repo_run.rglob("*")):
    if path.is_file():
        print(path.relative_to(REPO_DIR), path.stat().st_size, "bytes")

if MODE == "merge":
    final_summary = repo_run / "final" / "v06_confirmatory_holm.json"
    report = json.loads(final_summary.read_text(encoding="utf-8"))
    print(json.dumps(report["methods"], indent=2))
    print(json.dumps(report.get("multiple_testing", {}), indent=2))

In [ ]:
#@title 6. Optional secure GitHub commit and push
import getpass
import os
from pathlib import Path
import stat
import subprocess

should_push = PUSH_TO_GITHUB and (MODE != "dry_run" or PUSH_DRY_RUN)
if not should_push:
    print("GitHub push skipped by configuration.")
else:
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None

    if not token:
        token = getpass.getpass(
            "GitHub token (fine-grained token with Contents: Read and write): "
        )
    if not token:
        raise RuntimeError("A GitHub token is required for push.")

    active_run_id = f"{RUN_ID}-dry-run" if MODE == "dry_run" else RUN_ID
    relative_output = Path("results") / "colab" / active_run_id

    subprocess.run(["git", "add", str(relative_output)], check=True, cwd=REPO_DIR)
    staged = subprocess.run(
        ["git", "diff", "--cached", "--quiet"],
        cwd=REPO_DIR,
    ).returncode != 0

    if staged:
        message = f"results: save Colab {MODE} output for {active_run_id}"
        subprocess.run(["git", "commit", "-m", message], check=True, cwd=REPO_DIR)
    else:
        print("No new Git files to commit.")

    askpass = Path("/tmp/fedfalsify_git_askpass.sh")
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) echo "x-access-token" ;;\n'
        '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding="utf-8",
    )
    askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)

    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["GITHUB_TOKEN"] = token

    # Rebase in case another chunk was pushed from a different Colab session.
    subprocess.run(
        ["git", "pull", "--rebase", "origin", PUSH_BRANCH],
        check=True,
        cwd=REPO_DIR,
        env=env,
    )
    subprocess.run(
        ["git", "push", "origin", f"HEAD:{PUSH_BRANCH}"],
        check=True,
        cwd=REPO_DIR,
        env=env,
    )

    askpass.unlink(missing_ok=True)
    del token
    env.pop("GITHUB_TOKEN", None)
    print("Pushed result commit to:", PUSH_BRANCH)

## Required execution sequence

1. Keep `MODE="dry_run"` and run all cells once.
2. Set `MODE="primary_chunk"` and run with `CHUNK_INDEX=0`.
3. Repeat in the same or a new Colab session for indexes `1`, `2`, and `3`.
4. Set `MODE="merge"` after all four chunks are present.
5. Confirm that `COMPLETE` and the `final/` directory exist in both GitHub and
   Google Drive.

For GitHub push, add a Colab secret named `GITHUB_TOKEN` and allow notebook
access. Use a fine-grained token limited to this repository with
**Contents: Read and write** permission. The token is never written to the
repository or Drive.